In [2]:
import pandas as pd

df = pd.read_csv("../data/india_yearly_electricity.csv")
df.info()

df.columns = df.columns.str.strip()
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    df[col] = df[col].str.strip().str.lower()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17214 entries, 0 to 17213
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              17214 non-null  object 
 1   Country code         17214 non-null  object 
 2   State                17214 non-null  object 
 3   State code           17214 non-null  object 
 4   State type           17214 non-null  object 
 5   Year                 17214 non-null  int64  
 6   Category             17214 non-null  object 
 7   Subcategory          17214 non-null  object 
 8   Variable             17214 non-null  object 
 9   Unit                 17214 non-null  object 
 10  Value                17055 non-null  float64
 11  YoY absolute change  8659 non-null   float64
 12  YoY % change         5464 non-null   float64
dtypes: float64(3), int64(1), object(9)
memory usage: 1.7+ MB


In [3]:
display(df.head())
print("\nYears available:", sorted(df["Year"].unique().tolist()))
print("State types:", df["State type"].unique())
print("Subcategories:", df["Subcategory"].unique())
print("Units:", df["Unit"].unique())
print("Variables:", df["Variable"].unique())
print("Categories:", df["Category"].unique())


national_states = df[~df["State type"].isin(["state", "union territory"])]["State"].unique()
print("National-level State labels:", national_states)

,Country,Country code,State,State code,State type,Year,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,india,ind,andaman and nicobar,an,union territory,2019,capacity,aggregate fuel,clean,mw,17.44,NaN,NaN
1,india,ind,andaman and nicobar,an,union territory,2019,capacity,aggregate fuel,fossil,mw,40.05,NaN,NaN
2,india,ind,andaman and nicobar,an,union territory,2019,capacity,aggregate fuel,gas and other fossil,mw,40.05,NaN,NaN
3,india,ind,andaman and nicobar,an,union territory,2019,capacity,aggregate fuel,"hydro, bioenergy and other renewables",mw,5.25,NaN,NaN
4,india,ind,andaman and nicobar,an,union territory,2019,capacity,aggregate fuel,renewables,mw,17.44,NaN,NaN



Years available: [2019, 2020, 2021, 2022, 2023, 2024]
State types: ['union territory' 'state' 'total']
Subcategories: ['aggregate fuel' 'fuel' 'total' 'co2 intensity']
Units: ['mw' '%' 'gwh' 'ktco2' 'gco2/kwh']
Variables: ['clean' 'fossil' 'gas and other fossil'
 'hydro, bioenergy and other renewables' 'renewables' 'wind and solar'
 'bioenergy' 'coal' 'gas' 'hydro' 'nuclear' 'other fossil'
 'other renewables' 'solar' 'wind' 'total generation' 'co2 intensity'
 'total emissions']
Categories: ['capacity' 'electricity generation' 'power sector emissions']
National-level State labels: ['india total']


In [4]:
# 1. Filter to national totals and individual fuel sources (avoids double-counting aggregates)
india_df = df[(df["State"] == "india total") & (df["Subcategory"] == "fuel")].copy()

# 2. Verify available Category + Unit combinations
print(india_df[["Category", "Unit"]].drop_duplicates())

# 3. Reshape Capacity into a Year x Fuel table and convert MW -> GW
cap_raw = india_df[(india_df["Category"] == "capacity") & (india_df["Unit"] == "mw")]
cap_gw = cap_raw.pivot(index="Year", columns="Variable", values="Value") / 1000.0

# 4. Reshape Electricity Generation into a Year x Fuel table and convert GWh -> BU
gen_raw = india_df[(india_df["Category"] == "electricity generation") & (india_df["Unit"] == "gwh")]
gen_bu = gen_raw.pivot(index="Year", columns="Variable", values="Value") / 1000.0

print("\n--- Installed Capacity (GW) ---")
display(cap_gw.round(2))

print("\n--- Electricity Generation (BU) ---")
display(gen_bu.round(2))

                    Category   Unit
5910                capacity     mw
5931  electricity generation      %
5940  electricity generation    gwh
5963  power sector emissions      %
5972  power sector emissions  ktco2

--- Installed Capacity (GW) ---


Variable,bioenergy,coal,gas,hydro,nuclear,other fossil,other renewables,solar,wind
Year,,,,,,,,,
2019,10.00,205.11,24.90,50.07,6.78,0.64,0.0,33.78,37.51
2020,10.31,206.11,24.90,50.55,6.78,0.51,0.0,37.51,38.62
2021,10.61,209.81,24.87,51.35,6.78,0.51,0.0,49.43,40.07
2022,10.73,210.63,24.81,51.74,6.78,0.56,NaN,63.30,41.90
2023,10.84,214.42,25.03,51.86,7.48,0.56,NaN,73.32,44.74
2024,11.35,219.06,24.81,52.03,8.18,0.56,NaN,97.86,48.16



--- Electricity Generation (BU) ---


Variable,bioenergy,coal,gas,hydro,nuclear,other fossil,other renewables,solar,wind
Year,,,,,,,,,
2019,14.75,998.84,47.98,161.72,45.16,0.10,0.38,46.27,63.31
2020,14.50,947.53,52.17,163.36,44.61,0.13,1.21,58.68,60.43
2021,15.33,1072.84,39.38,159.99,43.92,0.10,2.16,68.31,68.09
2022,16.35,1164.74,25.16,174.51,46.19,0.18,2.50,95.16,70.05
2023,14.93,1265.17,29.40,148.75,48.20,0.33,2.67,113.41,82.11
2024,13.48,1327.96,33.93,155.80,54.70,0.36,2.86,133.80,81.54


In [5]:
# 1. Pivot Power Sector Emissions (ktCO2 -> Million Tonnes CO2)
em_raw = india_df[(india_df["Category"] == "power sector emissions") & (india_df["Unit"] == "ktco2")]
emissions_mt = em_raw.pivot(index="Year", columns="Variable", values="Value") / 1000.0

# 2. Clean missing capacity values (replaces NaN in 'other renewables' with 0.0)
''' Methodological Note: `other renewables`
In the Ember India dataset, separate installed capacity (`MW`) for `other renewables` is not reported after 2021 (`NaN` for 2022–2024),
while minor residual generation (`2.86 BU` in 2024, or `0.16%` of total electricity generation) continues to be logged. 
Rather than imputing synthetic capacity figures, `other renewables` is set to `0.0 GW` in capacity-driven projections (`2025–2040`).
'''
cap_gw_clean = cap_gw.fillna(0.0)

# 3. Calculate annual capacity additions (GW added each year)
annual_additions_gw = cap_gw_clean.diff()

# 4. Calculate historical Capacity Utilisation Factor (CUF %)
# Formula: Generation (BU) / (Capacity (GW) * 8.76 hours/yr) * 100
cuf_pct = (gen_bu / (cap_gw_clean.replace(0.0, float("nan")) * 8.76)) * 100

# 5. Extract per-fuel Emission Intensity in the background (MT CO2 per BU)
emission_factor = emissions_mt / gen_bu.replace(0.0, float("nan"))

# 6. Add a row-wise Total column on a separate copy so matrix shapes stay aligned
emissions_summary = emissions_mt.copy()
emissions_summary["total_mt_co2"] = emissions_summary.sum(axis=1)

print("--- Annual Capacity Additions (GW/yr) ---")
display(annual_additions_gw.round(2))

print("\n--- Historical Capacity Utilisation Factor (CUF %) ---")
display(cuf_pct.round(2))

print("\n--- Power Sector Emissions (MT CO2) ---")
display(emissions_summary.round(2))

--- Annual Capacity Additions (GW/yr) ---


Variable,bioenergy,coal,gas,hydro,nuclear,other fossil,other renewables,solar,wind
Year,,,,,,,,,
2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020,0.31,1.00,0.00,0.48,0.0,-0.13,0.0,3.73,1.12
2021,0.30,3.70,-0.03,0.80,0.0,0.00,0.0,11.92,1.44
2022,0.12,0.82,-0.06,0.39,0.0,0.05,0.0,13.87,1.83
2023,0.12,3.79,0.21,0.12,0.7,0.00,0.0,10.02,2.84
2024,0.50,4.64,-0.22,0.17,0.7,0.00,0.0,24.55,3.43



--- Historical Capacity Utilisation Factor (CUF %) ---


Variable,bioenergy,coal,gas,hydro,nuclear,other fossil,other renewables,solar,wind
Year,,,,,,,,,
2019,16.83,55.59,22.00,36.87,76.04,1.85,NaN,15.64,19.27
2020,16.04,52.48,23.92,36.89,75.12,2.88,NaN,17.86,17.86
2021,16.50,58.37,18.08,35.57,73.94,2.26,NaN,15.77,19.40
2022,17.40,63.12,11.57,38.50,77.78,3.60,NaN,17.16,19.09
2023,15.72,67.36,13.41,32.74,73.57,6.75,NaN,17.66,20.95
2024,13.56,69.20,15.61,34.18,76.34,7.37,NaN,15.61,19.33



--- Power Sector Emissions (MT CO2) ---


Variable,bioenergy,coal,gas,hydro,nuclear,other fossil,other renewables,solar,wind,total_mt_co2
Year,,,,,,,,,,
2019,3.39,819.04,23.51,3.88,0.54,0.07,0.01,2.22,0.70,853.37
2020,3.33,776.98,25.56,3.92,0.54,0.09,0.05,2.82,0.66,813.95
2021,3.53,879.73,19.30,3.84,0.53,0.07,0.08,3.28,0.75,911.10
2022,3.76,955.09,12.33,4.19,0.55,0.12,0.10,4.57,0.77,981.47
2023,3.43,1037.44,14.41,3.57,0.58,0.23,0.10,5.44,0.90,1066.11
2024,3.10,1088.92,16.63,3.74,0.66,0.25,0.11,6.42,0.90,1120.73


In [6]:
# 1. Extract 2024 starting capacity (GW) and 2024 total electricity generation (BU)
starting_gw = cap_gw_clean.iloc[-1]
starting_total_gen_bu = gen_bu.iloc[-1].sum()

# 2. Compute recent 3-year average (2022-2024) for annual additions and CUF
# Clamping negative tiny drift (like gas -0.02) to 0.0 for clean baseline expansion
recent_additions_gw = annual_additions_gw.tail(3).mean().clip(lower=0.0)
baseline_cuf_pct = cuf_pct.tail(3).mean().fillna(0.0)

# 3. Extract per-fuel emission factor (MT CO2 per BU)
baseline_ef = emission_factor.mean().fillna(0.0)

# 4. Combine all calibrated parameters into a single readable summary table
calibration_summary = pd.DataFrame({
    "2024_capacity_gw": starting_gw,
    "recent_additions_gw_yr": recent_additions_gw,
    "baseline_cuf_pct": baseline_cuf_pct,
    "emission_factor_mt_per_bu": baseline_ef
})

print(f"2024 Total Electricity Generation (Demand Anchor): {starting_total_gen_bu:.2f} BU\n")
display(calibration_summary.round(3))

# 5. Store as clean Python dictionaries for our simulation loop
baseline_params = calibration_summary.to_dict(orient="index")

2024 Total Electricity Generation (Demand Anchor): 1804.42 BU



,2024_capacity_gw,recent_additions_gw_yr,baseline_cuf_pct,emission_factor_mt_per_bu
Variable,,,,
bioenergy,11.348,0.246,15.559,0.230
coal,219.060,3.083,66.561,0.820
gas,24.808,0.000,13.532,0.490
hydro,52.033,0.228,35.142,0.024
nuclear,8.180,0.467,75.896,0.012
other fossil,0.562,0.018,5.905,0.700
other renewables,0.000,0.000,0.000,0.038
solar,97.865,16.145,16.808,0.048
wind,48.163,2.699,19.788,0.011


In [7]:
def simulate_grid(
    params,
    start_demand_bu,
    start_year=2025,
    end_year=2040,
    demand_growth_pct=6.0,
    addition_overrides=None
):
    """
    Simulates annual capacity (GW), electricity generation (BU),
    and power-sector emissions (MT CO2) from start_year to end_year.
    """
    if addition_overrides is None:
        addition_overrides = {}

    # Track active capacity (GW) per fuel without modifying original baseline_params
    current_cap = {
        fuel: values["2024_capacity_gw"]
        for fuel, values in params.items()
    }

    clean_fuels = {"solar", "wind", "hydro", "nuclear", "bioenergy", "other renewables"}
    demand_bu = start_demand_bu
    records = []

    for year in range(start_year, end_year + 1):
        # 1. Compound annual electricity demand growth
        demand_bu = demand_bu * (1.0 + demand_growth_pct / 100.0)

        total_gw = 0.0
        total_gen_bu = 0.0
        clean_gen_bu = 0.0
        total_co2_mt = 0.0

        # 2. Update stock-and-flow metrics for each fuel
        for fuel, values in params.items():
            annual_add = addition_overrides.get(fuel, values["recent_additions_gw_yr"])
            current_cap[fuel] = max(0.0, current_cap[fuel] + annual_add)

            gen_bu_fuel = current_cap[fuel] * (values["baseline_cuf_pct"] / 100.0) * 8.76
            co2_mt_fuel = gen_bu_fuel * values["emission_factor_mt_per_bu"]

            total_gw += current_cap[fuel]
            total_gen_bu += gen_bu_fuel
            total_co2_mt += co2_mt_fuel

            if fuel in clean_fuels:
                clean_gen_bu += gen_bu_fuel

        # 3. Store annual summary metrics
        records.append({
            "Year": year,
            "total_capacity_gw": total_gw,
            "solar_gw": current_cap["solar"],
            "wind_gw": current_cap["wind"],
            "coal_gw": current_cap["coal"],
            "total_gen_bu": total_gen_bu,
            "demand_bu": demand_bu,
            "supply_gap_bu": total_gen_bu - demand_bu,
            "clean_share_pct": (clean_gen_bu / total_gen_bu) * 100.0,
            "total_co2_mt": total_co2_mt,
            "grid_intensity_gco2_kwh": (total_co2_mt / total_gen_bu) * 1000.0
        })

    return pd.DataFrame(records).set_index("Year")


# Run Baseline Scenario (using 2022-2024 historical expansion rates & 6% annual demand growth)
baseline_sim = simulate_grid(
    params=baseline_params,
    start_demand_bu=starting_total_gen_bu,
    demand_growth_pct=6.0
)

print("--- Baseline Simulation (2025–2040) ---")
display(baseline_sim.round(2))

--- Baseline Simulation (2025–2040) ---


,total_capacity_gw,solar_gw,wind_gw,coal_gw,total_gen_bu,demand_bu,supply_gap_bu,clean_share_pct,total_co2_mt,grid_intensity_gco2_kwh
Year,,,,,,,,,,
2025,484.90,114.01,50.86,222.14,1815.17,1912.69,-97.52,27.01,1093.94,602.67
2026,507.79,130.15,53.56,225.23,1865.75,2027.45,-161.70,28.02,1110.02,594.94
2027,530.68,146.30,56.26,228.31,1916.33,2149.10,-232.77,28.98,1126.09,587.63
2028,553.56,162.44,58.96,231.39,1966.90,2278.04,-311.14,29.89,1142.16,580.69
2029,576.45,178.59,61.66,234.48,2017.48,2414.73,-397.24,30.76,1158.23,574.10
2030,599.33,194.73,64.35,237.56,2068.06,2559.61,-491.55,31.58,1174.31,567.83
2031,622.22,210.88,67.05,240.64,2118.64,2713.19,-594.55,32.37,1190.38,561.86
2032,645.10,227.02,69.75,243.73,2169.22,2875.98,-706.76,33.12,1206.45,556.17
2033,667.99,243.17,72.45,246.81,2219.80,3048.54,-828.74,33.83,1222.52,550.74


In [8]:
# 1. Define policy overrides for an Accelerated Clean Transition
accelerated_overrides = {
    "solar": 42.0,     # Accelerated utility + rooftop solar rollout (GW/yr)
    "wind": 9.0,       # Onshore + offshore wind expansion (GW/yr)
    "nuclear": 1.5,    # Faster reactor commissioning (GW/yr)
    "coal": -1.5       # No new coal plants + gradual retirement of aging units (GW/yr)
}

# 2. Run Accelerated Scenario (with efficiency gains moderating demand growth to 4.8%/yr)
accel_sim = simulate_grid(
    params=baseline_params,
    start_demand_bu=starting_total_gen_bu,
    demand_growth_pct=4.8,
    addition_overrides=accelerated_overrides
)

# 3. Build a side-by-side comparison table across key metrics
comparison_df = pd.concat([
    baseline_sim["clean_share_pct"].rename("base_clean_%"),
    accel_sim["clean_share_pct"].rename("accel_clean_%"),
    baseline_sim["total_co2_mt"].rename("base_co2_mt"),
    accel_sim["total_co2_mt"].rename("accel_co2_mt"),
    baseline_sim["grid_intensity_gco2_kwh"].rename("base_intensity"),
    accel_sim["grid_intensity_gco2_kwh"].rename("accel_intensity"),
    baseline_sim["supply_gap_bu"].rename("base_gap_bu"),
    accel_sim["supply_gap_bu"].rename("accel_gap_bu"),
], axis=1)

# 4. Print headline climate milestones
print(f"Baseline Peak Emissions:    {baseline_sim['total_co2_mt'].max():.2f} MT in {baseline_sim['total_co2_mt'].idxmax()}")
print(f"Accelerated Peak Emissions: {accel_sim['total_co2_mt'].max():.2f} MT in {accel_sim['total_co2_mt'].idxmax()}")
print(f"2040 Annual CO2 Avoided:    {baseline_sim.loc[2040, 'total_co2_mt'] - accel_sim.loc[2040, 'total_co2_mt']:.2f} MT/yr\n")

display(comparison_df.round(2))

Baseline Peak Emissions:    1335.03 MT in 2040
Accelerated Peak Emissions: 1074.06 MT in 2025
2040 Annual CO2 Avoided:    318.15 MT/yr



,base_clean_%,accel_clean_%,base_co2_mt,accel_co2_mt,base_intensity,accel_intensity,base_gap_bu,accel_gap_bu
Year,,,,,,,,
2025,27.01,29.61,1093.94,1074.06,602.67,582.36,-97.52,-46.73
2026,28.02,32.98,1110.02,1070.25,594.94,556.26,-161.70,-57.78
2027,28.98,36.08,1126.09,1066.44,587.63,532.22,-232.77,-73.19
2028,29.89,38.95,1142.16,1062.62,580.69,510.03,-311.14,-93.17
2029,30.76,41.60,1158.23,1058.81,574.10,489.47,-397.24,-117.93
2030,31.58,44.07,1174.31,1055.00,567.83,470.38,-491.55,-147.71
2031,32.37,46.36,1190.38,1051.19,561.86,452.59,-594.55,-182.74
2032,33.12,48.50,1206.45,1047.38,556.17,435.99,-706.76,-223.28
2033,33.83,50.51,1222.52,1043.56,550.74,420.45,-828.74,-269.60


In [9]:
import json
from pathlib import Path

# 1. Ensure the output directory exists
output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "modelData.json"

# 2. Assemble metadata, calibrated parameters, historical data, and scenario projections
model_data = {
    "metadata": {
        "base_year": 2024,
        "sim_start_year": 2025,
        "sim_end_year": 2040,
        "demand_anchor_bu": round(float(starting_total_gen_bu), 3),
        "units": {
            "capacity": "GW",
            "generation": "BU",
            "emissions": "MT CO2",
            "grid_intensity": "gCO2/kWh"
        },
        "methodological_note": (
            "other renewables lacks reported capacity after 2021 (2.86 BU or 0.16% of 2024 generation) "
            "and is set to 0.0 GW in capacity-driven projections rather than imputing synthetic capacity."
        )
    },
    "baseline_params": calibration_summary.round(4).to_dict(orient="index"),
    "historical": {
        "capacity_gw": cap_gw_clean.round(3).reset_index().to_dict(orient="records"),
        "generation_bu": gen_bu.fillna(0.0).round(3).reset_index().to_dict(orient="records"),
        "emissions_mt": emissions_mt.fillna(0.0).round(3).reset_index().to_dict(orient="records")
    },
    "scenarios": {
        "baseline": baseline_sim.round(3).reset_index().to_dict(orient="records"),
        "accelerated": accel_sim.round(3).reset_index().to_dict(orient="records")
    }
}

# 3. Write formatted JSON to disk
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(model_data, f, indent=2)

print(f"Exported successfully to: {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size / 1024:.2f} KB")
print("Top-level JSON keys:", list(model_data.keys()))

Exported successfully to: /home/pk/Downloads/india-grid-simulator/output/modelData.json
File size: 18.71 KB
Top-level JSON keys: ['metadata', 'baseline_params', 'historical', 'scenarios']
